# 11.2 自适应步长与误差控制

固定步长方法需要预先选择整段区间的步长。自适应方法在每一步估计局部误差，误差过大就拒绝并缩小步长，误差足够小就接受并尝试放大步长。

In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    heun_euler_embedded_step,
    solve_ivp_adaptive_heun,
    solve_ivp_fixed_step,
)

## Heun-Euler 嵌入式对

同一个右端函数采样可以同时给出 Euler 低阶值和 Heun 高阶值。二者差值提供局部误差估计。

In [ ]:
def exponential_rhs(_time, state):
    return state

high, low, error = heun_euler_embedded_step(exponential_rhs, 0.0, 1.0, 0.1)
high, low, error

## 自适应求解

缩放误差范数小于 1 时接受步长；否则拒绝并缩小。下面用 $y'=y$ 检查误差目标。

In [ ]:
adaptive = solve_ivp_adaptive_heun(
    exponential_rhs,
    (0.0, 1.0),
    1.0,
    initial_step=0.25,
    absolute_tolerance=1e-7,
    relative_tolerance=1e-5,
)
adaptive.values[-1, 0], math.e, adaptive.accepted_steps, adaptive.rejected_steps

In [ ]:
np.column_stack([adaptive.times[1:], adaptive.step_sizes, adaptive.error_estimates])[:8]

## 拒绝过大的初始步长

如果初始步长过大，误差估计会触发一次或多次拒绝。

In [ ]:
def fast_rhs(_time, state):
    return 10.0 * state

fast = solve_ivp_adaptive_heun(
    fast_rhs,
    (0.0, 0.2),
    1.0,
    initial_step=0.2,
    absolute_tolerance=1e-8,
    relative_tolerance=1e-6,
)
fast.rejected_steps, fast.accepted_steps, fast.values[-1, 0]

## 与固定步长对照

自适应步长不必把最小必要步长用于整段区间；固定步长则需要人工选择保守步长。

In [ ]:
fixed = solve_ivp_fixed_step(exponential_rhs, (0.0, 1.0), 1.0, 0.02, method="heun")
fixed.values[-1, 0], fixed.times.size - 1, adaptive.accepted_steps

## 小结

* 嵌入式方法用同一步内的两个阶数估计局部误差。
* 缩放误差范数把绝对误差和相对误差要求合并。
* 接受/拒绝机制让求解器在平缓区域增大步长，在快速变化区域缩小步长。